# Luka's Log

In [1]:
import json
offical_run = json.load(open("LSC22.json"))
# 'id', 'name', 'description', 'started', 'ended', 'tasks', 'hasStarted', 'running', 'hasEnded'

In [2]:
descriptions = offical_run['description']
# 'id', 'name', 'description', 'taskTypes', 'taskGroups', 'tasks', 'teams', 'teamGroups', 'participantCanView'
tasks = descriptions['tasks']
# list of ['id', 'name', 'taskGroup', 'taskType', 'duration', 'mediaCollectionId', 'target', 'hints']
targets = {}
queries = {}
for i in range(len(tasks)):
    if "KIS" in tasks[i]['name']:
        targets[tasks[i]['name']] = [item['location'] for item in tasks[i]['target']['items']]
        queries[tasks[i]['name']] = [item['text'] for item in tasks[i]['hints']]
    else:
        targets[tasks[i]['name']] = []
        queries[tasks[i]['name']] = [item['text'] for item in tasks[i]['hints']]
print(len(targets), len(queries))

45 45


In [3]:
teams = offical_run['description']['teams']
# list of ['uid', 'name', 'color', 'logoId', 'users']
team_ids = {item["uid"]["string"]: item["name"] for item in teams}

In [4]:
TEAMS = list(team_ids.values())

In [5]:
tasks = offical_run['tasks']
print("Number of tasks: ", len(tasks))
# list of ['started', 'ended', 'submissions', 'description', 'filter', 'scorer', 'validator', 'duration', 'uid',
#          'taskDescriptionId', 'position', 'hasStarted', 'running', 'hasEnded', 'teamGroupAggregators']
# ['teamId', 'memberId', 'timestamp', 'item', 'uid', 'status']
submissions = {}
start_times = {}
mysceal = []
names = []
real_queries= {}
real_targets = {}
for i in range(len(tasks)):
    name = tasks[i]["description"]["name"]
    if name in queries and tasks[i]['submissions']:
        names.append(name)
        real_queries[name] = queries[name]
        real_targets[name] = targets[name]
        start_times[name] = tasks[i]["started"]
        submissions[name] = [(team_ids[item["teamId"]["string"]],
                           item["status"],
                         (item["timestamp"] - start_times[name])/1000,
                          item["item"]["name"]) for item in tasks[i]['submissions']]


Number of tasks:  26


In [6]:
TASKS = ["KIS", "QA", "AD"]

def task_type(name):
    if "QA" in name:
        return "QA"
    elif "KIS" in name:
        return "KIS"
    return "AD"

def make_task_dicts(data_type):
    # deep copy obj to 3 keys: AD, QA, KIS
    return {task: defaultdict(data_type) for task in TASKS}

In [7]:
# Find the task with no submissions from MySceal
for name in submissions:
    found = False
    for team, *_ in submissions[name]:
        if team == 'MyScéal':
            found= True
            break
    if not found:
        print(name)

In [8]:
# Task distribution:
from collections import Counter

task_dist = Counter()

for task in names:
    task = task_type(task)
    task_dist[task] += 1

print(task_dist)

Counter({'KIS': 10, 'QA': 9, 'AD': 6})


## Correct/Incorrect

In [9]:
from collections import defaultdict
correct_counts = make_task_dicts(int)
incorrect_counts = make_task_dicts(int)
total_counts = make_task_dicts(int)
time_till_correct = defaultdict(lambda: defaultdict(lambda: None))
time_till_correct_full = defaultdict(lambda: defaultdict(lambda: 300))
top_3 = make_task_dicts(int)
limits = {"QA": 180, "KIS": 300, "AD": 180}

for team in TEAMS:
    for name in names:
        time_till_correct_full[team][name] = limits[task_type(name)]

for name in submissions:
    correct_time = 0
    task = task_type(name) 
    for team, status, time, image in submissions[name]:
        if status == "CORRECT":
            correct_counts[task][team] += 1
            time = min(time, limits[task])
            if time_till_correct[team][name]:
                time_till_correct[team][name] = min(time_till_correct[team][name], time)
            else:
                time_till_correct[team][name] = time
            
            time_till_correct_full[team][name] = min(time_till_correct_full[team][name], time)

            correct_time += 1
            if correct_time <= 3:
                top_3[task][team] += 1
        else:
            incorrect_counts[task][team] += 1

        total_counts[task][team] += 1

### Scoring

In [10]:
import math
import numpy as np
# Scores for each queries
max_point = 100
max_point_end = 50
penalty = 10
scores = {"QA": [],
          "AD": [],
          "KIS": []}
recall_ad = defaultdict(list)
precision_ad = defaultdict(list)

for name, submission in submissions.items():
    task = task_type(name)
    if task == "AD":
        correct = defaultdict(int)
        incorrect = defaultdict(int)
        score = defaultdict(float)
        num_correct = set()
        penalty = 0.2
        
        max_time = 180
        for team, status, time, image in submission:
            if status == "WRONG":
                incorrect[team] += 1
            elif status == "CORRECT":
                correct[team] += 1
                num_correct.add(image)
                real_targets[name].append(image)
                
        for team in TEAMS:
            if correct[team] == 0:
                score[team] = 0
                recall_ad[team].append(0)
                precision_ad[team].append(0)
            else:
                score[team] = correct[team] * max_point / (correct[team] + incorrect[team]/2) * correct[team] / len(num_correct)
                recall_ad[team].append(correct[team] / len(num_correct))
                precision_ad[team].append(correct[team] / (correct[team] + incorrect[team]))
    else:
        score = defaultdict(float)
        for team, status, time, image in submission:
            if status == "WRONG":
                score[team] -= penalty
            elif status == "CORRECT":
                score[team] += max_point_end + (max_point - max_point_end) * (1 - time/limits[task])
        for team in TEAMS:
            score[team] = max(0, score[team])
    scores[task].append(score)

precision_ad = {team: np.mean(precision_ad[team]) for team in TEAMS}
recall_ad = {team: np.mean(recall_ad[team]) for team in TEAMS}


In [12]:
# Export to json real tasks and their targets
data = []
for i in range(len(names)):
    data.append(
        {
            "name": names[i],
            "queries": real_queries[names[i]],
            "targets": real_targets[names[i]],
            "submissions": submissions[names[i]],
        }
    )
json.dump(data, open("LSC22_tasks.json", "w"), indent=2)

In [ ]:
from collections import defaultdict, OrderedDict
TASKS = ["KIS", "AD", "QA"]
normalize_scores = {"QA": {},
                    "AD": {},
                    "KIS": {},
                    "SUM": defaultdict(float)}
for task in TASKS:
    cum_scores = defaultdict(float)
    for score in scores[task]: # iterate over all tasks in this task_type
        for team in score:
            cum_scores[team] += score[team]
                
    max_score = max(cum_scores.values())
    for team in cum_scores:
        normalize_scores[task][team] = round(cum_scores[team]/max_score * 100)
        normalize_scores["SUM"][team] += normalize_scores[task][team]
        
# sort TEAMS by normalize_scores
TEAMS = sorted(TEAMS, key=lambda team: normalize_scores["SUM"][team], reverse=True)
for task in TASKS:
    # sort dict by TEAMS
    normalize_scores[task] = OrderedDict(sorted(normalize_scores[task].items(), key=lambda x: TEAMS.index(x[0])))
print([(team, normalize_scores["SUM"][team]) for team in TEAMS])

In [ ]:
normalize_scores["AD"]

# GRAPHS

In [ ]:
import pandas as pd
import seaborn as sns

# Each row is a task corresponding to a team
# Columns: team, task, task_type, score, incorrect, correct, time_till_correct
detailed_df = pd.DataFrame(
    columns=[
        "team",
        "task",
        "task_type",
        "score",
        "incorrect",
        "correct",
        "time_till_correct",
    ]
)
for name, submission in submissions.items():
    task = task_type(name)
    correct = defaultdict(int)
    incorrect = defaultdict(int)
    if task == "AD":
        score = defaultdict(float)
        num_correct = set()
        penalty = 0.2
        max_time = 180
        for team, status, time, image in submission:
            if status == "WRONG":
                incorrect[team] += 1
            elif status == "CORRECT":
                correct[team] += 1
                num_correct.add(image)

        for team in TEAMS:
            if correct[team] == 0:
                score[team] = 0
            else:
                score[team] = (
                    correct[team]
                    * max_point
                    / (correct[team] + incorrect[team] / 2)
                    * correct[team]
                    / len(num_correct)
                )
    else:
        score = defaultdict(float)
        for team, status, time, image in submission:
            if status == "WRONG":
                score[team] -= penalty
                incorrect[team] += 1
            elif status == "CORRECT":
                score[team] += max_point_end + (max_point - max_point_end) * (
                    1 - time / limits[task]
                )
                correct[team] += 1
        for team in TEAMS:
            score[team] = max(0, score[team])
    for team in TEAMS:
        detailed_df = detailed_df.append(
            {
                "team": team,
                "task": name,
                "task_type": task,
                "score": score[team],
                "incorrect": incorrect[team],
                "correct": correct[team],
                "time_till_correct": time_till_correct[team][name],
            },
            ignore_index=True,
        )
detailed_df.to_csv("detailed_scores.csv")

In [ ]:
import pandas as pd
import seaborn as sns

df = None
for task in TASKS:
    data = {
        "team": TEAMS,
        "task": [f"{task}" for team in TEAMS],
        f"correct": [correct_counts[task][team] for team in TEAMS],
        f"incorrect": [incorrect_counts[task][team] for team in TEAMS],
        # f"cum_score_{task}": [cum_scores[task][team] for team in TEAMS],
        f"norm_score": [normalize_scores[task][team] for team in TEAMS],
    }
    if task != "AD":
        data.update({
            f"top_3": [top_3[task][team] for team in TEAMS],
            f"precision": [correct_counts[task][team]/total_counts[task][team] for team in TEAMS],
            f"recall": [correct_counts[task][team]/len(scores[task]) for team in TEAMS]})
        
    else:
        data["recall"] = [recall_ad[team] for team in TEAMS]
        data["precision"] = [precision_ad[team] for team in TEAMS]
    for i in time_till_correct["LifeSeeker"]:
        data[f"time_correct_{i}"] = [time_till_correct[team][i] for team in TEAMS]
    for i in time_till_correct_full["LifeSeeker"]:
        data[f"time_correct_full_{i}"] = [time_till_correct_full[team][i] for team in TEAMS]
    if df is None:
        df = pd.DataFrame(data=data)
    else:
        df = pd.concat([df, pd.DataFrame(data=data)], axis=0)      
df

In [ ]:
# change QA to KIS-QA
df["task"] = df["task"].str.replace("QA", "KIS-QA")

In [ ]:
df.to_csv('lsc22.csv', index=False)

In [ ]:
task_names = ["KIS", "KIS-QA", "AD"]

import matplotlib.pyplot as plt

sns.set_style("white")
plt.rcParams.update({'font.size': 16})
all_score = df.groupby(['team', 'task']).sum().reset_index().pivot(index='team', columns='task', values='norm_score')

# Sort task by ["KIS", "AD", "AD"]
all_score = all_score.reindex(columns=task_names)

# sort by the other in TEAMS
all_score = all_score.loc[TEAMS[::-1]]
fig = all_score.plot.barh(figsize=(15, 5), rot=0, stacked=True, 
                    title="Overall scores in LSC'22", width=0.8,
                    color=["#98D2EB", "#B2B1CF", "#b0c5aa"], legend=True)

fig.bar_label(fig.containers[0], labels=all_score["KIS"], label_type='center')
fig.bar_label(fig.containers[1], labels=all_score["KIS-QA"], label_type='center')
fig.bar_label(fig.containers[2], labels=all_score["AD"], label_type='center')

plt.savefig("overall_22.png", format="png", bbox_inches='tight')

In [ ]:
# !pip install -U matplotlib
import matplotlib.pyplot as plt
sns.set_style("white")
plt.rcParams.update({'font.size': 16})

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(15, 8))

# Plot data for each TASK
for i, TASK in enumerate(["KIS", "KIS-QA"]):
    filtered = df[df["task"] == TASK]
    ax = filtered[["team", "correct", "incorrect"]].plot(x='team', kind='bar', stacked=True, width=0.8, color=["#8da0cb", "#fc8d62"], ax=axes[i])
    ax.bar_label(ax.containers[0], labels=filtered["correct"], label_type='center')
    ax.bar_label(ax.containers[1], labels=["" if x == 0 else x for x in filtered["incorrect"]], label_type='center')
    
    if i == 0:
        # set y ticks to be integers
        from matplotlib.ticker import MaxNLocator
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    
    
    
    ax.set_xlabel("Teams")
    # ax.set_ylabel("Number of {} queries".format(TASK), fontweight='bold')
    ax.set_xticklabels(filtered["team"])
    if i == 0:
        ax.text(-0.12, 0.5, "Number of Submissions", va='center', rotation='vertical', transform=ax.transAxes)

    # make y-axis ends at 12
    # ax.set_ylim(0, 12)
    
    # Turn off legend
    ax.legend().set_visible(False)

    # Title of the subplot
    ax.set_title(f"{TASK}", fontweight='bold')



# Create a single legend for all subplots
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles[::-1], ["Incorrect", "Correct"], loc='upper right', bbox_to_anchor=(1, 0.965), frameon=False, prop={'weight': 'bold'})
plt.subplots_adjust(top=0.9)  # Adjust the top margin for the suptitle
fig.suptitle("Number of incorrect and correct queries per team in LSC'22 in KIS and KIS-QA tasks.", fontweight='bold')

# Add spacing between subplots
plt.tight_layout()
# Show the plot
plt.show()

print()
# plt.savefig("incorrect_correct.eps", format="eps", bbox_inches='tight', dpi=1200)
fig.savefig("incorrect_correct_22.png", format="png", bbox_inches='tight')

In [ ]:
# !pip install -U matplotlib
import matplotlib.pyplot as plt
sns.set_style("white")
plt.rcParams.update({'font.size': 16})

# set figure size
plt.rcParams["figure.figsize"] = (8, 7)

# Plot data for each TASK
filtered = df[df["task"] == "AD"]
fig = filtered[["team", "precision", "recall"]].plot(x='team', kind='bar', width=0.8, color=["#8da0cb", "#fc8d62"])
fig.bar_label(fig.containers[0], labels=filtered["precision"].round(2), label_type='edge', fontsize=12)
fig.bar_label(fig.containers[1], labels=["" if x == 0 else x for x in filtered["recall"].round(2)], label_type='edge',fontsize=11)

plt.xlabel("Team", fontweight='bold')
# ax.set_ylabel("Number of {} queries".format(TASK), fontweight='bold')f
# make y-axis ends at 12
# ax.set_ylim(0, 12)

# Turn off legend
fig.legend().set_visible(False)

# Create a single legend for all subplots
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, ["Precision", "Recall"], loc='upper right', bbox_to_anchor=(1, 1), frameon=False, prop={'weight': 'bold'})
plt.subplots_adjust(top=0.9)  # Adjust the top margin for the suptitle
plt.title("Precision and Recall per team in LSC'22 for Ad-hoc tasks.", fontweight='bold')

# Add spacing between subplots
plt.tight_layout()
# Show the plot
plt.show()

print()
# plt.savefig("incorrect_correct.eps", format="eps", bbox_inches='tight', dpi=1200)
plt.savefig("precision-recall_22.png", format="png", bbox_inches='tight')


In [ ]:
# !pip install -U matplotlib
import matplotlib.pyplot as plt
sns.set_style("white")
plt.rcParams.update({'font.size': 16})

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(15, 8))

# Plot data for each TASK
for i, TASK in enumerate(["KIS", "KIS-QA"]):
    filtered = df[df["task"] == TASK]
    if TASK == "KIS-QA":
        time_name="QA"
    else: 
        time_name=TASK
    ax = filtered[[f"time_correct_{i}" for i in time_till_correct["LifeSeeker"] if task_type(i) == time_name]].T.plot(kind='box',
        showmeans=False, showfliers=True,
        boxprops=dict(facecolor="#8da0cb", color="black", linewidth=1),
        medianprops=dict(color="black", linewidth=1),
        whiskerprops=dict(color="black", linewidth=1),
        capprops=dict(color="black", linewidth=1),
        flierprops=dict(marker='o', markersize=6, color="#8da0cb"),
        patch_artist=True,
        ax=axes[i])
    
    # ax.bar_label(ax.containers[0], labels=filtered["precision"], label_type='center')
    # ax.bar_label(ax.containers[1], labels=["" if x == 0 else x for x in filtered["recall"]], label_type='center')
    
    ax.set_xlabel("Teams")
    # ax.set_ylabel("Number of {} queries".format(TASK), fontweight='bold')
    ax.set_xticklabels(filtered["team"])
    if i == 0:
        ax.text(-0.12, 0.5, "Seconds", va='center', rotation='vertical', transform=ax.transAxes)

    # set rotation of x-axis labels
    ax.tick_params(axis='x', rotation=90)
    
    # add a horizontal line for the limit (with label)
    ax.axhline(y=limits[time_name], color='r', linestyle='-', label=f"Time limit: {limits[time_name]}s")

    if time_name == "KIS":
        ax.set_ylim(0, 310)
    elif time_name == "QA":
        ax.set_ylim(0, 205)
        # set ticks to be 0, 20, 40, 60, 80, 100, 120, 140, 160, 180
        ax.set_yticks(np.arange(0, 220, 20))
    else:
        ax.set_ylim(0, 205)
        ax.set_yticks(np.arange(0, 220, 20))
    
    # Turn off legend
    ax.legend().set_visible(False)

    # Title of the subplot
    ax.set_title(f"{TASK}", fontweight='bold')
    


# Create a single legend for all subplots
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, ["Time limit"], loc='upper right', bbox_to_anchor=(1, 1), frameon=False, prop={'weight': 'bold'})
plt.subplots_adjust(top=0.9)  # Adjust the top margin for the suptitle
fig.suptitle("Time to find a correct submission per team in LSC'22", fontweight='bold')

# Add spacing between subplots
plt.tight_layout()
# Show the plot
plt.show()

print()
# plt.savefig("incorrect_correct.eps", format="eps", bbox_inches='tight', dpi=1200)
fig.savefig("time_22.png", format="png", bbox_inches='tight')

In [ ]:
# !pip install -U matplotlib
import matplotlib.pyplot as plt
sns.set_style("white")
plt.rcParams.update({'font.size': 16})

# Create subplots
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# Plot data for each TASK
for i, TASK in enumerate(TASKS):
    filtered = df[df["task"] == TASK]
    ax = filtered[[f"time_correct_full_{i}" for i in time_till_correct["LifeSeeker"] if task_type(i) == TASK]].T.plot(kind='box',
        showmeans=False, showfliers=True,
        boxprops=dict(facecolor="#8da0cb", color="black", linewidth=1),
        medianprops=dict(color="black", linewidth=1),
        whiskerprops=dict(color="black", linewidth=1),
        capprops=dict(color="black", linewidth=1),
        flierprops=dict(marker='o', markersize=6, color="#8da0cb"),
        patch_artist=True,
        ax=axes[i])
    
    # ax.bar_label(ax.containers[0], labels=filtered["precision"], label_type='center')
    # ax.bar_label(ax.containers[1], labels=["" if x == 0 else x for x in filtered["recall"]], label_type='center')
    # add a horizontal line for the limit (with label)
    ax.axhline(y=limits[TASK], color='r', linestyle='-', label=f"Time limit: {limits[TASK]}s")

    if TASK == "KIS":
        ax.set_ylim(0, 310)
    elif TASK == "QA":
        ax.set_ylim(0, 200)
    else:
        ax.set_ylim(0, 200)
    
    ax.set_xlabel("Teams")
    # ax.set_ylabel("Number of {} queries".format(TASK), fontweight='bold')
    ax.set_xticklabels(filtered["team"])
    if i == 0:
        ax.text(-0.12, 0.5, "Seconds", va='center', rotation='vertical', transform=ax.transAxes)

    # set rotation of x-axis labels
    ax.tick_params(axis='x', rotation=90)
    
    # Turn off legend
    ax.legend().set_visible(False)

    # Title of the subplot
    ax.set_title(f"{TASK}", fontweight='bold')

# Create a single legend for all subplots
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, ["Time limit"], loc='upper right', bbox_to_anchor=(1, 1), frameon=False, prop={'weight': 'bold'})
plt.subplots_adjust(top=0.9)  # Adjust the top margin for the suptitle
fig.suptitle("Time to find a correct submission per team in LSC'22", fontweight='bold')

# Add spacing between subplots
plt.tight_layout()
# Show the plot
plt.show()

print()
# plt.savefig("incorrect_correct.eps", format="eps", bbox_inches='tight', dpi=1200)
fig.savefig("time_full_22.png", format="png", bbox_inches='tight')